# SOM-OT Warp Benchmark
## Head-to-head comparison against ComBat, CytoNormPy, cyCombine, and MMD-ResNet

**Dataset**: Trussart / CytofRUV (FR-FCM-Z2L2)  
- 2 batches (RUV1b, RUV3b) — same samples measured in different CyTOF runs  
- 5 patients × 4 gated cell types (A, B, C, D) + Unassigned  
- 33 protein markers, ~8.6M total cells  
- **All 5 samples are shared** between batches → any sample can serve as anchor  

### Two evaluation tracks

| Track | Training data | What it tests | Methods |
|-------|--------------|---------------|---------|
| **A — With anchor** | Anchor sample only (sample "1" in both batches) | Proper reference-based correction | SOM-OT, CytoNorm |
| **B — Without anchor** | All data (no shared reference used) | Correction without reference samples | SOM-OT, ComBat, MMD-ResNet, cyCombine |

Both tracks evaluate on the **same full dataset** using the same metrics, including  
**biological preservation** (cell type silhouette, graph connectivity).

In [23]:
import sys, os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
%matplotlib inline
%load_ext autoreload
%autoreload 2

REPO = Path(os.environ.get('SOMOT_REPO', Path.cwd().resolve().parent))  # run from scripts/
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / 'scripts'))

from benchmark_public import (
    load_trussart, compute_all_metrics, per_marker_emd,
    knn_batch_entropy, knn_batch_ilisi, graph_connectivity,
    run_somot, run_combat, run_mmdresnet
)
from somot_warp.sklearn_api import SOMOTWarpSKExact
print('Imports OK')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Imports OK


---
## 1. Load Dataset & Identify Anchor Samples

The Trussart/CytofRUV dataset has pre-gated FCS files organized by batch (RUV1b, RUV3b),  
with cell types encoded in file prefixes (A–D) and sample IDs as numeric suffixes (1–5).  
All 5 samples were measured in both batches — we use **sample "1"** as the anchor.

In [24]:
# ── Configuration ──
DATA_DIR = os.environ.get('SOMOT_TRUSSART_DIR', str(REPO / 'data' / 'trussart'))
ANCHOR_BATCH  = 'RUV1b'       # anchor batch (correction maps non-anchor → anchor scale)
ANCHOR_SAMPLE = '1'            # shared sample ID for anchor-based training
SUBSAMPLE     = 20000          # cells per batch for speed (0 = all ~8.6M cells)

# ── Load data using benchmark_public loader ──
X_full, y_batch_full, y_sample_full, y_celltype_full, markers = load_trussart(DATA_DIR)
print(f'\nLoaded: {X_full.shape[0]:,} cells  ×  {len(markers)} markers')
print(f'Batches: {dict(zip(*np.unique(y_batch_full, return_counts=True)))}')
print(f'Cell types: {dict(zip(*np.unique(y_celltype_full, return_counts=True)))}')

  Loading cached data from <SOMOT_TRUSSART_DIR>/_cache_trussart.parquet

Loaded: 8,621,862 cells  ×  33 markers
Batches: {'RUV1b': np.int64(5726616), 'RUV3b': np.int64(2895246)}
Cell types: {'A': np.int64(2665479), 'B': np.int64(3259600), 'C': np.int64(2674695), 'D': np.int64(12084), 'Unassigned': np.int64(10004)}


In [25]:
# ── Verify anchor sample is in both batches ──
anchor_mask_full = y_sample_full == ANCHOR_SAMPLE
anchor_batches = np.unique(y_batch_full[anchor_mask_full])
print(f'Anchor sample "{ANCHOR_SAMPLE}" found in batches: {list(anchor_batches)}')
assert len(anchor_batches) >= 2, f'Anchor must be in >=2 batches, found: {anchor_batches}'

print(f'\nAnchor cells per batch:')
for b in anchor_batches:
    n = ((y_sample_full == ANCHOR_SAMPLE) & (y_batch_full == b)).sum()
    print(f'  {b}: {n:,}')

# ── Subsample for speed ──
if SUBSAMPLE > 0:
    rng = np.random.RandomState(42)
    keep = []
    for b in np.unique(y_batch_full):
        idx = np.where(y_batch_full == b)[0]
        n = min(SUBSAMPLE, len(idx))
        keep.append(rng.choice(idx, n, replace=False))
    keep = np.sort(np.concatenate(keep))
    X        = X_full[keep].copy()
    y_batch  = y_batch_full[keep].copy()
    y_sample = y_sample_full[keep].copy()
    y_celltype = y_celltype_full[keep].copy()
    print(f'\nSubsampled: {len(X):,} cells  ({SUBSAMPLE:,} per batch)')
else:
    X = X_full; y_batch = y_batch_full; y_sample = y_sample_full; y_celltype = y_celltype_full

is_anchor = y_sample == ANCHOR_SAMPLE
print(f'Anchor cells in subset: {is_anchor.sum():,}  ({is_anchor.sum()/len(X)*100:.1f}%)')

# Drop Unassigned cells for cleaner evaluation (optional)
keep_ct = y_celltype != 'Unassigned'
X = X[keep_ct]; y_batch = y_batch[keep_ct]; y_sample = y_sample[keep_ct]
y_celltype = y_celltype[keep_ct]; is_anchor = is_anchor[keep_ct]
print(f'After dropping Unassigned: {len(X):,} cells')
print(f'X shape: {X.shape}')

Anchor sample "1" found in batches: ['RUV1b', 'RUV3b']

Anchor cells per batch:
  RUV1b: 1,566,689
  RUV3b: 922,703

Subsampled: 40,000 cells  (20,000 per batch)
Anchor cells in subset: 11,890  (29.7%)
After dropping Unassigned: 39,947 cells
X shape: (39947, 33)


In [26]:
# ── Quick data summary ──
print(f'{"="*60}')
print(f'DATASET SUMMARY')
print(f'{"="*60}')
print(f'Total cells:  {len(X):,}')
print(f'Markers:      {len(markers)}')
print(f'Batches:      {dict(zip(*np.unique(y_batch, return_counts=True)))}')
print(f'Cell types:   {dict(zip(*np.unique(y_celltype, return_counts=True)))}')
print(f'Samples:      {sorted(np.unique(y_sample))}')
print(f'Anchor cells: {is_anchor.sum():,} (sample "{ANCHOR_SAMPLE}")')
print(f'\nMarkers:')
for i, m in enumerate(markers):
    print(f'  {i+1:2d}. {m}')

DATASET SUMMARY
Total cells:  39,947
Markers:      33
Batches:      {'RUV1b': np.int64(19981), 'RUV3b': np.int64(19966)}
Cell types:   {'A': np.int64(12265), 'B': np.int64(13854), 'C': np.int64(13771), 'D': np.int64(57)}
Samples:      ['1', '2', '3', '4', '5']
Anchor cells: 11,890 (sample "1")

Markers:
   1. 89Y_CD45
   2. 115In_HLA-DR
   3. 139La
   4. 140Ce_CD27_MT271
   5. 141Pr_CD235ab
   6. 142Nd_CD19
   7. 143Nd_CD5
   8. 144Nd_CD38
   9. 145Nd_CD4
  10. 146Nd_CD8
  11. 147Sm_CD20
  12. 148Nd_CD16
  13. 151Eu_CD123
  14. 153Eu_Bcl-xL
  15. 154Sm_Bax_1B4
  16. 155Gd_CD56
  17. 156Gd_CD14
  18. 157Gd_Bcl-2
  19. 159Tb_CD11c
  20. 160Gd_Mcl-1
  21. 161Dy_cMyc
  22. 163Dy_BFL-1
  23. 165Ho_Bim
  24. 166Er_pRb
  25. 167Er_CD27
  26. 168Er_PUMA
  27. 169Tm_CD45RA
  28. 170Er_CD3
  29. 171Yb_CD66
  30. 172Yb_BCL-W
  31. 173Yb_cCaspase3
  32. 174Yb_p53
  33. 209Bi_CD61


---
## 2. Uncorrected Baseline

In [27]:
print('Computing uncorrected (baseline) metrics...')
metrics_before = compute_all_metrics(X, y_batch, bio_labels=y_celltype, markers=markers)

print(f"  kNN entropy:       {metrics_before['knn_entropy']:.4f}  (higher = better batch mixing)")
print(f"  iLISI:             {metrics_before['knn_ilisi']:.4f}  (higher = better batch mixing)")
print(f"  Silhouette batch:  {metrics_before['silhouette_batch']:.4f}  (lower = better batch mixing)")
print(f"  Silhouette bio:    {metrics_before.get('silhouette_bio', 'N/A')}  (higher = better bio preservation)")
print(f"  Graph connectivity:{metrics_before.get('graph_connectivity', 'N/A')}  (higher = better)")
print(f"  Median EMD:        {metrics_before['per_marker_emd']['median_EMD']:.4f}  (lower = better alignment)")

Computing uncorrected (baseline) metrics...
  kNN entropy:       0.6355  (higher = better batch mixing)
  iLISI:             1.4897  (higher = better batch mixing)
  Silhouette batch:  0.0181  (lower = better batch mixing)
  Silhouette bio:    -0.010551905259490013  (higher = better bio preservation)
  Graph connectivity:0.7820122340741636  (higher = better)
  Median EMD:        2.9157  (lower = better alignment)


---
# TRACK A: With Anchor Samples

**Design**: Fit correction on the anchor sample (sample "1", present in both RUV1b and RUV3b) only — where all distributional differences are purely technical batch effects — then apply the learned transform to the full dataset.

This is the gold-standard approach when reference samples are available. CytoNorm was designed for exactly this setting.

In [28]:
results_A = {}       # Track A results (with anchor)
corrected_A = {}     # Track A corrected data

### A1. SOM-OT Warp (trained on anchor only)

In [29]:
print('Track A — SOM-OT Warp (anchor-trained)...')
try:
    # Fit on anchor sample only (sample present in both batches)
    X_anchor = X[is_anchor]
    y_anchor = y_batch[is_anchor]
    print(f'  Training on anchor sample "{ANCHOR_SAMPLE}": {X_anchor.shape[0]:,} cells')
    print(f'  Anchor batch distribution: {dict(zip(*np.unique(y_anchor, return_counts=True)))}')

    model_A = SOMOTWarpSKExact(
        anchor='A', output_scale='A',
        transform_kind='arcsinh', pca_components=None,
        som_grid=(8, 8), som_iters=8000,
        n_landmarks_total=1200,
        q_grid=np.array([0, 1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 99, 100]),
        alpha_ot=0.7, alpha_warp=None,
        som_verbose=True,
    )
    # Map batch labels: anchor batch → 'A', other batch → 'B'
    y_anchor_mapped = np.where(y_anchor == ANCHOR_BATCH, 'A', 'B')

    t0 = time.perf_counter()
    model_A.fit(X_anchor, y_anchor_mapped)

    # Transform ALL data using the anchor-trained model
    y_all_mapped = np.where(y_batch == ANCHOR_BATCH, 'A', 'B')
    X_somot_A = model_A.transform(X, y_all_mapped)
    t_A = time.perf_counter() - t0
    print(f'  Done in {t_A:.1f}s')

    metrics_somot_A = compute_all_metrics(X_somot_A, y_batch, bio_labels=y_celltype, markers=markers)
    results_A['somot_anchor'] = {
        'display_name': 'SOM-OT (anchor)', 'elapsed_sec': t_A,
        'before': metrics_before, 'after': metrics_somot_A
    }
    corrected_A['somot_anchor'] = X_somot_A
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()

Track A — SOM-OT Warp (anchor-trained)...
  Training on anchor sample "1": 11,890 cells
  Anchor batch distribution: {'RUV1b': np.int64(5496), 'RUV3b': np.int64(6394)}
 [ 8000 / 8000 ] 100% - 0:00:00 left 
 quantization error: 3.8425495920986914


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 159.24it/s]


  Done in 1.0s


### A2. CytoNormPy (trained on anchor only)

In [30]:
print('Track A — CytoNormPy (anchor-trained)...')
try:
    from benchmark_public import run_cytonorm
    t0 = time.perf_counter()
    # CytoNormPy uses is_reference flag internally — samples in both batches = reference
    X_cn_A = run_cytonorm(X.copy(), y_batch.copy(), markers, ANCHOR_BATCH, y_sample=y_sample)
    t_cn = time.perf_counter() - t0
    print(f'  Done in {t_cn:.1f}s')

    metrics_cn_A = compute_all_metrics(X_cn_A, y_batch, bio_labels=y_celltype, markers=markers)
    results_A['cytonorm_anchor'] = {
        'display_name': 'CytoNorm (anchor)', 'elapsed_sec': t_cn,
        'before': metrics_before, 'after': metrics_cn_A
    }
    corrected_A['cytonorm_anchor'] = X_cn_A
except ImportError as e:
    print(f'  SKIPPED (not installed): {e}')
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()

Track A — CytoNormPy (anchor-trained)...
normalized file RUV1b_3
normalized file RUV3b_3
normalized file RUV1b_4
normalized file RUV1b_1
normalized file RUV1b_5
normalized file RUV1b_2
normalized file RUV3b_1
normalized file RUV3b_2
normalized file RUV3b_4
normalized file RUV3b_5
  Done in 1.8s


### Track A Summary

In [31]:
def make_summary(results_dict, metrics_before):
    """Build a summary DataFrame from a results dict."""
    def _row(label, d, time_str='-'):
        row = {
            'Method': label, 'Time (s)': time_str,
            'kNN Entropy ↑': f"{d.get('knn_entropy', 0):.4f}",
            'iLISI ↑': f"{d.get('knn_ilisi', 0):.4f}",
            'Sil. Batch ↓': f"{d.get('silhouette_batch', 0):.4f}",
            'Median EMD ↓': f"{d.get('per_marker_emd', {}).get('median_EMD', 0):.4f}",
        }
        if 'silhouette_bio' in d:
            row['Sil. Bio ↑'] = f"{d['silhouette_bio']:.4f}"
        if 'graph_connectivity' in d:
            row['Graph Conn. ↑'] = f"{d['graph_connectivity']:.4f}"
        return row

    rows = [_row('Uncorrected', metrics_before)]
    for key, res in results_dict.items():
        after = res.get('after', {})
        if not after:
            continue
        rows.append(_row(res['display_name'], after, f"{res['elapsed_sec']:.1f}"))
    return pd.DataFrame(rows)

print('═' * 60)
print('TRACK A: WITH ANCHOR SAMPLES')
print('═' * 60)
summary_A = make_summary(results_A, metrics_before)
summary_A

════════════════════════════════════════════════════════════
TRACK A: WITH ANCHOR SAMPLES
════════════════════════════════════════════════════════════


,Method,Time (s),kNN Entropy ↑,iLISI ↑,Sil. Batch ↓,Median EMD ↓,Sil. Bio ↑,Graph Conn. ↑
0,Uncorrected,-,0.6355,1.4897,0.0181,2.9157,-0.0106,0.7820
1,SOM-OT (anchor),1.0,0.7607,1.6311,0.0077,2.8434,-0.0177,0.7691
2,CytoNorm (anchor),1.8,0.8408,1.7206,0.0019,0.8334,-0.0159,0.7868


---
# TRACK B: Without Anchor Samples

**Design**: Train correction on the full dataset — no shared reference samples used.  
This tests each method's ability to correct batch effects from the joint distribution alone.  

ComBat, cyCombine, and MMD-ResNet only operate in this mode. SOM-OT Warp can also work here.

In [32]:
results_B = {}       # Track B results (no anchor)
corrected_B = {}     # Track B corrected data

### B1. SOM-OT Warp (trained on all data, no anchor)

In [33]:
print('Track B — SOM-OT Warp (no anchor)...')
try:
    t0 = time.perf_counter()
    X_somot_B = run_somot(X.copy(), y_batch.copy(), markers, ANCHOR_BATCH)
    t_B = time.perf_counter() - t0
    print(f'  Done in {t_B:.1f}s')

    metrics_somot_B = compute_all_metrics(X_somot_B, y_batch, bio_labels=y_celltype, markers=markers)
    results_B['somot_noanchor'] = {
        'display_name': 'SOM-OT (no anchor)', 'elapsed_sec': t_B,
        'before': metrics_before, 'after': metrics_somot_B
    }
    corrected_B['somot_noanchor'] = X_somot_B
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()

Track B — SOM-OT Warp (no anchor)...
 [ 8000 / 8000 ] 100% - 0:00:00 left 
 quantization error: 4.071650001706816


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:01<00:00, 39.57it/s]


  Done in 2.7s


### B2. ComBat (no anchor)

In [34]:
print('Track B — ComBat...')
try:
    t0 = time.perf_counter()
    X_combat = run_combat(X.copy(), y_batch.copy(), markers, ANCHOR_BATCH)
    t_cb = time.perf_counter() - t0
    print(f'  Done in {t_cb:.1f}s')

    metrics_combat = compute_all_metrics(X_combat, y_batch, bio_labels=y_celltype, markers=markers)
    results_B['combat'] = {
        'display_name': 'ComBat', 'elapsed_sec': t_cb,
        'before': metrics_before, 'after': metrics_combat
    }
    corrected_B['combat'] = X_combat
except ImportError as e:
    print(f'  SKIPPED (not installed): {e}')
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()

Track B — ComBat...
  Done in 0.1s


### B3. MMD-ResNet (no anchor)

In [35]:
print('Track B — MMD-ResNet...')
try:
    t0 = time.perf_counter()
    X_mmd = run_mmdresnet(X.copy(), y_batch.copy(), markers, ANCHOR_BATCH)
    t_mmd = time.perf_counter() - t0
    print(f'  Done in {t_mmd:.1f}s')

    metrics_mmd = compute_all_metrics(X_mmd, y_batch, bio_labels=y_celltype, markers=markers)
    results_B['mmdresnet'] = {
        'display_name': 'MMD-ResNet', 'elapsed_sec': t_mmd,
        'before': metrics_before, 'after': metrics_mmd
    }
    corrected_B['mmdresnet'] = X_mmd
except ImportError as e:
    print(f'  SKIPPED (not installed): {e}')
except Exception as e:
    print(f'  FAILED: {e}')
    import traceback; traceback.print_exc()

Track B — MMD-ResNet...
  Done in 3.8s


### B4. cyCombine (optional, requires R)

In [36]:
RUN_CYCOMBINE = False   # set True if R + cyCombine + rpy2 are installed

if RUN_CYCOMBINE:
    print('Track B — cyCombine...')
    try:
        from benchmark_public import run_cycombine
        t0 = time.perf_counter()
        X_cyc = run_cycombine(X.copy(), y_batch.copy(), markers, ANCHOR_BATCH)
        t_cyc = time.perf_counter() - t0
        print(f'  Done in {t_cyc:.1f}s')

        metrics_cyc = compute_all_metrics(X_cyc, y_batch, bio_labels=y_celltype, markers=markers)
        results_B['cycombine'] = {
            'display_name': 'cyCombine', 'elapsed_sec': t_cyc,
            'before': metrics_before, 'after': metrics_cyc
        }
        corrected_B['cycombine'] = X_cyc
    except Exception as e:
        print(f'  FAILED: {e}')
else:
    print('cyCombine skipped (set RUN_CYCOMBINE = True if R + cyCombine + rpy2 are installed)')

cyCombine skipped (set RUN_CYCOMBINE = True if R + cyCombine + rpy2 are installed)


### Track B Summary

In [37]:
print('═' * 60)
print('TRACK B: WITHOUT ANCHOR SAMPLES')
print('═' * 60)
summary_B = make_summary(results_B, metrics_before)
summary_B

════════════════════════════════════════════════════════════
TRACK B: WITHOUT ANCHOR SAMPLES
════════════════════════════════════════════════════════════


,Method,Time (s),kNN Entropy ↑,iLISI ↑,Sil. Batch ↓,Median EMD ↓,Sil. Bio ↑,Graph Conn. ↑
0,Uncorrected,-,0.6355,1.4897,0.0181,2.9157,-0.0106,0.7820
1,SOM-OT (no anchor),2.7,0.9059,1.8119,0.0009,0.5823,-0.0146,0.7775
2,ComBat,0.1,0.6043,1.4637,0.0038,2.2345,-0.0158,0.7694
3,MMD-ResNet,3.8,0.5753,1.4306,0.0187,2.9475,-0.0109,0.7689


---
# Combined Results

## 3. Side-by-Side Summary Table

In [ ]:
# ── Focus on Track B only (no anchor) for paper figures ──
# Track A (anchor-trained) is omitted from the paper discussion.
all_results = {}
all_corrected = {}
for k, v in results_B.items():
    all_results[k] = v
    if k in corrected_B:
        all_corrected[k] = corrected_B[k]

# Rename "SOM-OT (no anchor)" → "SOM-OT-Warp" since anchor mode is not discussed
if 'somot_noanchor' in all_results:
    all_results['somot_noanchor']['display_name'] = 'SOM-OT-Warp'

# Build combined table
def _combined_row(label, d, time_str='-'):
    row = {
        'Method': label, 'Time (s)': time_str,
        'kNN Entropy ↑': d.get('knn_entropy', 0),
        'iLISI ↑': d.get('knn_ilisi', 0),
        'Sil. Batch ↓': d.get('silhouette_batch', 0),
        'Median EMD ↓': d.get('per_marker_emd', {}).get('median_EMD', 0),
    }
    if 'silhouette_bio' in d:
        row['Sil. Bio ↑'] = d['silhouette_bio']
    if 'graph_connectivity' in d:
        row['Graph Conn. ↑'] = d['graph_connectivity']
    return row

rows = [_combined_row('Uncorrected', metrics_before)]
for key, res in all_results.items():
    after = res.get('after', {})
    if not after:
        continue
    rows.append(_combined_row(res['display_name'], after, f"{res['elapsed_sec']:.1f}"))

combined_df = pd.DataFrame(rows)
for c in combined_df.columns:
    if combined_df[c].dtype in [float, np.float64]:
        combined_df[c] = combined_df[c].round(4)
combined_df

---
## 4. Comparison Plots

### 4a. Batch Mixing Metrics (all methods, both tracks)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

plot_metrics = [
    ('knn_entropy', 'kNN Entropy (↑ better)'),
    ('knn_ilisi', 'iLISI (↑ better)'),
    ('silhouette_batch', 'Sil. Batch (↓ better)'),
    ('silhouette_bio', 'Sil. Bio (↑ better)'),
]

method_colors = {
    'somot_noanchor': '#6495ED',   # cornflower blue
    'combat':         '#F08080',   # light coral
    'mmdresnet':      '#90EE90',   # light green
}

for ax, (mkey, title) in zip(axes, plot_metrics):
    names = ['Uncorrected']
    vals = [metrics_before.get(mkey, 0)]
    colors = ['#B0B0B0']

    for key, res in all_results.items():
        after = res.get('after', {})
        if mkey in after:
            names.append(res['display_name'])
            vals.append(after[mkey])
            colors.append(method_colors.get(key, '#999999'))

    bars = ax.barh(range(len(names)), vals, color=colors)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(str(REPO / 'results' / 'trussart' / 'fig_bench_metrics.png'), dpi=200, bbox_inches='tight')
plt.show()

### 4b. Per-Marker EMD Heatmap (% reduction)

In [ ]:
emd_before = metrics_before['per_marker_emd']['per_marker']

reduction_rows = []
for key, res in all_results.items():
    after = res.get('after', {})
    if not after:
        continue
    emd_after = after.get('per_marker_emd', {}).get('per_marker', {})
    label = res['display_name']
    for m in markers:
        bv = emd_before.get(m, 0)
        av = emd_after.get(m, 0)
        reduction_rows.append({
            'Method': label, 'Marker': m,
            'EMD_reduction': (bv - av) / (bv + 1e-12)
        })

if reduction_rows:
    red_df = pd.DataFrame(reduction_rows)
    pivot = red_df.pivot(index='Marker', columns='Method', values='EMD_reduction')
    pivot = pivot.loc[markers]

    fig, ax = plt.subplots(figsize=(max(6, len(all_results) * 2.5),
                                    max(8, len(markers) * 0.35)))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'EMD Reduction'},
                annot_kws={'fontsize': 8})
    ax.set_title('EMD Reduction per Marker  (green = improved, red = worsened)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.savefig(str(REPO / 'results' / 'trussart' / 'fig_bench_emd_heatmap.png'), dpi=200, bbox_inches='tight')
    plt.show()

### 4c. Runtime Comparison

In [ ]:
method_colors = {
    'somot_noanchor': '#6495ED',
    'combat':         '#F08080',
    'mmdresnet':      '#90EE90',
}

fig, ax = plt.subplots(figsize=(8, max(2.5, len(all_results) * 0.7)))
names = []; times = []; colors = []
for key, res in all_results.items():
    if not res.get('after'):
        continue
    names.append(res['display_name'])
    times.append(res['elapsed_sec'])
    colors.append(method_colors.get(key, '#999999'))

bars = ax.barh(names, times, color=colors)
ax.set_xlabel('Seconds')
ax.set_title('Runtime (fit + transform)', fontsize=12, fontweight='bold')
for bar, t in zip(bars, times):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
            f'{t:.1f}s', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(str(REPO / 'results' / 'trussart' / 'fig_bench_runtime.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## 5. Per-Marker Distribution Overlay

In [ ]:
n_show = min(6, len(markers))
show_markers = markers[:n_show]

# Columns: Uncorrected + Track B methods only
col_names = ['Uncorrected']
col_data  = [X]
for key, res in all_results.items():
    if not res.get('after'):
        continue
    col_names.append(res['display_name'])
    col_data.append(all_corrected[key])

batches = np.unique(y_batch)
fig, axes = plt.subplots(n_show, len(col_names),
                         figsize=(3.5 * len(col_names), 2.2 * n_show),
                         sharex='row')
if n_show == 1: axes = axes[np.newaxis, :]
if len(col_names) == 1: axes = axes[:, np.newaxis]

for col, (cname, Xm) in enumerate(zip(col_names, col_data)):
    for row, mk in enumerate(show_markers):
        ax = axes[row, col]
        j = markers.index(mk)
        for b in batches:
            mask = y_batch == b
            ax.hist(Xm[mask, j], bins=80, density=True, histtype='step', label=b, linewidth=1)
        if row == 0: ax.set_title(cname, fontsize=10, fontweight='bold')
        if col == 0: ax.set_ylabel(mk, fontsize=9)
        if row == 0 and col == len(col_names) - 1: ax.legend(fontsize=7)
        ax.tick_params(labelsize=6)

plt.suptitle('Per-Marker Distributions by Batch', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(str(REPO / 'results' / 'trussart' / 'fig_bench_distributions.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## 6. UMAP Visualization

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler

n_umap = min(15000, len(X))
rng = np.random.RandomState(42)
umap_idx = rng.choice(len(X), n_umap, replace=False)
batch_sub = y_batch[umap_idx]
ct_sub = y_celltype[umap_idx]

# Only Track B methods (no anchor)
umap_datasets = {'Uncorrected': X}
for key, res in all_results.items():
    if res.get('after') and key in all_corrected:
        umap_datasets[res['display_name']] = all_corrected[key]

umap_embeddings = {}
for name, data in umap_datasets.items():
    print(f'UMAP: {name}...')
    Xs = StandardScaler().fit_transform(data[umap_idx])
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42, verbose=False)
    umap_embeddings[name] = reducer.fit_transform(Xs)
print('Done.')

In [ ]:
n_plots = len(umap_embeddings)

# ── Row 1: colored by Batch ──
fig, axes = plt.subplots(1, n_plots, figsize=(4.5 * n_plots, 4))
if n_plots == 1: axes = [axes]

batch_colors = {b: plt.cm.Set1(i) for i, b in enumerate(np.unique(batch_sub))}
for ax, (name, emb) in zip(axes, umap_embeddings.items()):
    for b in np.unique(batch_sub):
        mask = batch_sub == b
        ax.scatter(emb[mask, 0], emb[mask, 1], s=0.3, alpha=0.25,
                   label=b, color=batch_colors[b])
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=10, fontsize=7)
plt.suptitle('UMAP colored by Batch', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(REPO / 'results' / 'trussart' / 'fig_umap_batch.png'), dpi=200, bbox_inches='tight')
plt.show()

# ── Row 2: colored by Cell Type ──
fig2, axes2 = plt.subplots(1, n_plots, figsize=(4.5 * n_plots, 4))
if n_plots == 1: axes2 = [axes2]

ct_colors = {ct: plt.cm.tab10(i) for i, ct in enumerate(sorted(np.unique(ct_sub)))}
for ax, (name, emb) in zip(axes2, umap_embeddings.items()):
    for ct in sorted(np.unique(ct_sub)):
        mask = ct_sub == ct
        ax.scatter(emb[mask, 0], emb[mask, 1], s=0.3, alpha=0.25,
                   label=ct, color=ct_colors[ct])
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=10, fontsize=7, title='Cell type')
plt.suptitle('UMAP colored by Cell Type', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(str(REPO / 'results' / 'trussart' / 'fig_umap_celltype.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## 7. Save All Results

In [45]:
OUT_DIR = REPO / 'results' / 'trussart'
OUT_DIR.mkdir(parents=True, exist_ok=True)

combined_df.to_csv(OUT_DIR / 'summary_combined.csv', index=False)
summary_A.to_csv(OUT_DIR / 'summary_trackA_anchor.csv', index=False)
summary_B.to_csv(OUT_DIR / 'summary_trackB_noanchor.csv', index=False)

for key, res in all_results.items():
    method_dir = OUT_DIR / key
    method_dir.mkdir(exist_ok=True)
    with open(method_dir / 'metrics.json', 'w') as f:
        json.dump({
            'elapsed_sec': res.get('elapsed_sec', 0),
            'before': res.get('before', {}),
            'after': res.get('after', {})
        }, f, indent=2, default=str)
    if key in all_corrected:
        pd.DataFrame(all_corrected[key], columns=markers).to_parquet(
            method_dir / 'corrected.parquet'
        )

print(f'All results saved to {OUT_DIR}/')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(OUT_DIR)}')

All results saved to <repo>/results/trussart/
  combat/corrected.parquet
  combat/metrics.json
  cytonorm_anchor/corrected.parquet
  cytonorm_anchor/metrics.json
  mmdresnet/corrected.parquet
  mmdresnet/metrics.json
  somot_anchor/corrected.parquet
  somot_anchor/metrics.json
  somot_noanchor/corrected.parquet
  somot_noanchor/metrics.json
  summary_combined.csv
  summary_trackA_anchor.csv
  summary_trackB_noanchor.csv
